# TSFM-PDM Industrial Benchmarking — Unified Colab Notebook
**Author:** Yassire Ammouri  
**Paper:** *Benchmarking Time-Series Foundation Models for Industrial Predictive Maintenance* (ICATH 2025)

---

| Step | Content |
|------|---------|
| 1 | Environment setup — GPU check, clone repo, install packages |
| 2 | Dataset download — C-MAPSS (NASA/Kaggle), Wind SCADA (Kaggle), MIMII (Zenodo) |
| 3 | Data preprocessing — chronological splits, per-sensor normalization |
| 4 | **Zero-shot experiments** — MOMENT · Chronos · Lag-Llama · PatchTST × 3 datasets |
| 5 | **Few-shot LoRA** — MOMENT + Lag-Llama × 3 datasets (1 % training data, r=16) |
| 6 | **Cross-condition transfer** — C-MAPSS FD001 → FD001 / FD002 / FD003 / FD004 |
| 7 | Results aggregation — DataFrames + pivot tables |
| 8 | Inference benchmarking + CSV export of all results |
| 9 | Figures — bar charts, ZS vs FS comparison, cross-condition heatmap |
| 10 | Export everything to Google Drive |

**Estimated runtime on Colab Free T4:** 10–15 hours  
> **Before running:** Runtime → Change runtime type → **T4 GPU**

## ⚙️ Section 0 — Configuration
Edit the variables below before executing the notebook.

In [ ]:
# ================================================================
# USER CONFIGURATION — edit before running
# ================================================================
from google.colab import userdata
# GitHub repository URL (set after you push the repo)
GITHUB_REPO = "https://github.com/Yassire1/icath-article.git"

# Google Drive folder to save checkpoints + results (leave empty to skip)
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/tsfm_pdm_bench/results"

# Kaggle API credentials (needed for Wind SCADA + optional C-MAPSS download)
# Alternatively upload kaggle.json to /root/.kaggle/ via Files panel
KAGGLE_USERNAME = "ammouriyassire"
KAGGLE_KEY = userdata.get('kaggle_api')

# Experiments to run
RUN_ZERO_SHOT       = True
RUN_FEW_SHOT        = True
RUN_CROSS_CONDITION = True

# Models
MODELS_ZERO_SHOT = ["moment", "chronos", "lag_llama", "patchtst"]
MODELS_FEW_SHOT  = ["moment", "lag_llama"]   # only LoRA-capable models

# Datasets
DATASETS = ["cmapss", "wind_scada", "mimii"]

# C-MAPSS subsets (FD001 always required; add FD002-FD004 for cross-condition)
CMAPSS_SUBSETS = ["FD001", "FD002", "FD003", "FD004"]

# Preprocessing
CMAPSS_LOOKBACK  = 64    # shorter context fits short engine sequences (min 128 cycles)
CMAPSS_HORIZON   = 30    # ~30-step ahead sensor forecasting
LOOKBACK         = 512   # Wind SCADA + MIMII
HORIZON          = 96

# Few-shot LoRA
LORA_R           = 16
LORA_ALPHA       = 32
LORA_EPOCHS      = 10
LORA_LR          = 1e-4
TRAIN_RATIO      = 0.01  # 1 % of training data

# Misc
SEED       = 42
DEVICE     = "cuda"
USE_WANDB  = False
WANDB_PROJECT = "tsfm-pdm-bench"


## 🖥️ Section 1 — Environment Setup

In [ ]:
# 1-A  GPU check
import subprocess, sys, os

result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(result.stdout[:600] if result.returncode == 0 else "nvidia-smi not available")

import torch
if torch.cuda.is_available():
    DEVICE = "cuda"
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    DEVICE = "cpu"
    print("WARNING: No GPU found — CPU inference will be very slow.")


In [ ]:
# 1-B  Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_AVAILABLE = True
    print(f"Drive mounted. Results will be copied to: {DRIVE_RESULTS_DIR}")
except Exception:
    DRIVE_AVAILABLE = False
    print("Not in Colab / Drive not mounted. Results saved locally only.")


In [ ]:
# 1-C  Clone repository
REPO_DIR = "/content/tsfm-pdm-bench"

if not os.path.exists(REPO_DIR):
    if "YOUR_USERNAME" in GITHUB_REPO:
        raise ValueError(
            "Please set GITHUB_REPO in the config cell above.\n"
            "Example: GITHUB_REPO = 'https://github.com/yourusername/tsfm-pdm-bench.git'"
        )
    print(f"Cloning {GITHUB_REPO} …")
    subprocess.run(["git", "clone", GITHUB_REPO, REPO_DIR], check=True)
else:
    print(f"Repo already at {REPO_DIR} — pulling latest …")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], capture_output=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f"Working directory: {os.getcwd()}")


In [ ]:
# 1-D  Install Python dependencies  (5-10 min on first run)
import subprocess, sys

def pip_install(args):
    return subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + args, check=False)

# Keep the scientific stack on one NumPy ABI to avoid
# "numpy.dtype size changed" binary incompatibility errors.
core_stack = [
    '--upgrade', '--force-reinstall',
    'numpy<2.0', 'pandas', 'scipy', 'scikit-learn',
]
pip_install(core_stack)

pkgs = [
    "momentfm",
    "chronos-forecasting>=1.3.0",
    "gluonts>=0.15.0", "pytorch-lightning>=2.0.0",
    "peft>=0.12.0", "transformers>=4.44.0", "accelerate>=0.33.0", "huggingface_hub",
    "neuralforecast>=1.7.0",
    "librosa", "soundfile",
    "kaggle",
    "wandb", "torchmetrics",
    "numpy<2.0",
]
pip_install(['--upgrade'] + pkgs)

# Lag-Llama from GitHub
pip_install(
    ['--upgrade', 'git+https://github.com/time-series-foundation-models/lag-llama.git']
)

# Reinstall core stack at the end so compiled extensions
# definitely match the active NumPy version.
pip_install(core_stack)

print("Package installation finished.")
print()
print("*** IMPORTANT: Runtime -> Restart session, then re-run all cells from the top. ***")
print("    (compiled wheels were reinstalled; interpreter restart is required)")


In [ ]:
# 1-E  Verify imports + set seeds
import importlib.util, random, subprocess, sys
import numpy as np

try:
    import pandas as pd
    import scipy
    import sklearn
except Exception as e:
    raise RuntimeError(
        "Binary dependency import failed. Re-run the install dependencies cell, restart runtime, and run all cells from the top.\n"
        f"Original error: {e}"
    )

print(
    f"Versions -> numpy={np.__version__}, pandas={pd.__version__}, scipy={scipy.__version__}, sklearn={sklearn.__version__}"
)

critical_modules = {
    "momentfm": "momentfm",
    "chronos": "chronos-forecasting",
    "neuralforecast": "neuralforecast",
}

def _find_missing(module_names):
    return [m for m in module_names if importlib.util.find_spec(m) is None]

missing_critical = _find_missing(critical_modules.keys())
if missing_critical:
    print("Missing critical packages detected. Attempting repair...")
    for mod in missing_critical:
        pkg = critical_modules[mod]
        print(f"  Installing {pkg} ...")
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', pkg], check=False)

    # Re-check and try a no-deps fallback to avoid forcing NumPy upgrades.
    still_missing = _find_missing(critical_modules.keys())
    if still_missing:
        print("Retrying with --no-deps for remaining packages...")
        for mod in still_missing:
            pkg = critical_modules[mod]
            print(f"  Installing {pkg} (--no-deps) ...")
            subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', pkg], check=False)

    still_missing = _find_missing(critical_modules.keys())
    if still_missing:
        missing_pkgs = [critical_modules[m] for m in still_missing]
        raise RuntimeError(
            "Could not import required packages after repair: "
            + ", ".join(missing_pkgs)
            + ". Re-run Cell 8, restart runtime, then run Cell 9 again."
        )
    print("Critical package repair completed.")

packages_check = {
    "torch":           "PyTorch",
    "momentfm":        "MOMENT (momentfm)",
    "chronos":         "Chronos",
    "lag_llama":       "Lag-Llama",
    "neuralforecast":  "NeuralForecast (PatchTST)",
    "peft":            "PEFT / LoRA",
    "gluonts":         "GluonTS",
    "librosa":         "librosa (MIMII audio)",
    "wandb":           "WandB",
}
print("Package status:")
for pkg, label in packages_check.items():
    ok = importlib.util.find_spec(pkg) is not None
    print(f"  {'✓' if ok else '✗'} {label}")

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print(f"\nSeed: {SEED}  |  Device: {DEVICE}")


## 📦 Section 2 — Data Download

### Dataset sources
| Dataset | Size | Download method |
|---------|------|-----------------|
| **C-MAPSS** (NASA Turbofan) | ~2 MB | NASA portal **or** Kaggle (auto) |
| **Wind SCADA** | ~5 MB | Kaggle API (auto, needs credentials) |
| **MIMII** (machine sound) | ~1 GB per machine | Zenodo auto-download |

### C-MAPSS — two options
**Option A (recommended):** set `KAGGLE_USERNAME` / `KAGGLE_KEY` above → auto-downloaded below.  
**Option B (manual):** download `CMAPSSData.zip` from  
https://ti.arc.nasa.gov/tech/dash/groups/pcoe/prognostic-data-repository/#turbofan  
then upload/copy to `/content/tsfm-pdm-bench/data/raw/cmapss/`.


In [ ]:
# 2-A  Create all raw data directories
from pathlib import Path

RAW_DIR  = Path("data/raw")
PROC_DIR = Path("data/processed")
RES_DIR  = Path("results")

for d in [
    RAW_DIR / "cmapss",
    RAW_DIR / "wind_scada",
    RAW_DIR / "mimii",
    PROC_DIR / "cmapss",
    PROC_DIR / "wind_scada",
    PROC_DIR / "mimii",
    RES_DIR / "zero_shot",
    RES_DIR / "few_shot",
    RES_DIR / "cross_condition",
    RES_DIR / "tables",
    RES_DIR / "figures",
]:
    d.mkdir(parents=True, exist_ok=True)

print("Directory tree created.")


In [ ]:
# 2-B  Configure Kaggle API
import os, json
from pathlib import Path

KAGGLE_DIR = Path("/root/.kaggle")
KAGGLE_DIR.mkdir(exist_ok=True)
kaggle_json = KAGGLE_DIR / "kaggle.json"

if not kaggle_json.exists():
    if KAGGLE_USERNAME and KAGGLE_KEY:
        kaggle_json.write_text(
            json.dumps({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY})
        )
        kaggle_json.chmod(0o600)
        print("Kaggle credentials written from config variables.")
    else:
        print("WARNING: Kaggle credentials not set.")
        print("  Either set KAGGLE_USERNAME / KAGGLE_KEY in the config cell,")
        print("  or upload kaggle.json via Files panel → /root/.kaggle/kaggle.json")
else:
    print("Kaggle credentials found at /root/.kaggle/kaggle.json")


In [ ]:
# 2-C  Download C-MAPSS (Kaggle, dataset: behrad3d/nasa-cmaps)
import subprocess
from pathlib import Path

CMAPSS_RAW = Path("data/raw/cmapss")
sentinel = CMAPSS_RAW / "train_FD001.txt"

if sentinel.exists():
    print("C-MAPSS already downloaded.")
else:
    print("Downloading C-MAPSS from Kaggle (behrad3d/nasa-cmaps) …")
    try:
        result = subprocess.run(
            ["kaggle", "datasets", "download", "-d", "behrad3d/nasa-cmaps",
             "-p", str(CMAPSS_RAW), "--unzip"],
            capture_output=True, text=True, check=True
        )
        print("Download complete.")
        # flatten any sub-folder
        import shutil as _shutil
        for sub in CMAPSS_RAW.iterdir():
            if sub.is_dir():
                all_moved = True
                for f in sub.iterdir():
                    dest = CMAPSS_RAW / f.name
                    try:
                        _shutil.move(str(f), str(dest))
                    except Exception as move_err:
                        print(f"  WARNING: could not move {f.name}: {move_err}")
                        all_moved = False
                if all_moved:
                    try:
                        sub.rmdir()
                    except OSError as rmdir_err:
                        print(f"  WARNING: could not remove dir {sub.name}: {rmdir_err}")
    except Exception as e:
        print(f"Kaggle download failed: {e}")

        print("\nManual fallback:")
        print("  1. Download CMAPSSData.zip from https://ti.arc.nasa.gov/c/6/")

        print("  2. Extract into  data/raw/cmapss/")

        print("  3. Files needed: train_FD001.txt … train_FD004.txt,")
        print("                   test_FD001.txt  … test_FD004.txt,")
        print("                   RUL_FD001.txt   … RUL_FD004.txt")

files = sorted(CMAPSS_RAW.glob("*.txt"))
print(f"\nC-MAPSS files ({len(files)}):", [f.name for f in files])

In [ ]:
# 2-D  Download Wind SCADA (Kaggle)
from pathlib import Path
import shutil
import subprocess

SCADA_RAW = Path("data/raw/wind_scada")
SCADA_RAW.mkdir(parents=True, exist_ok=True)
sentinel = next(SCADA_RAW.glob("*.csv"), None)

if sentinel:
    print(f"Wind SCADA already downloaded: {sentinel.name}")
else:
    print("Downloading Wind SCADA from Kaggle ...")

    # Kaggle slugs can change over time; try current known slug first.
    candidate_slugs = [
        "berkerisen/wind-turbine-scada-dataset",
        "berkerisen/wind-power-forecasting",
    ]

    download_ok = False
    last_error = ""

    for slug in candidate_slugs:
        print(f"  Trying dataset: {slug}")
        cmd = [
            "kaggle", "datasets", "download",
            "-d", slug,
            "-p", str(SCADA_RAW),
            "--unzip",
        ]
        proc = subprocess.run(cmd, capture_output=True, text=True)

        if proc.returncode == 0:
            print(f"  Download complete from {slug}.")
            download_ok = True
            break

        # Keep short diagnostic output to understand auth/not-found errors.
        err_text = (proc.stderr or proc.stdout or "").strip()
        last_error = err_text
        print(f"  Failed for {slug} (exit code {proc.returncode}).")
        if err_text:
            print("  Kaggle message:")
            for line in err_text.splitlines()[:8]:
                print(f"    {line}")

    if download_ok:
        # Flatten nested folders, if Kaggle extracted into a subdirectory.
        for sub in SCADA_RAW.iterdir():
            if sub.is_dir():
                all_moved = True
                for f in sub.iterdir():
                    dest = SCADA_RAW / f.name
                    try:
                        shutil.move(str(f), str(dest))
                    except Exception as move_err:
                        print(f"  WARNING: could not move {f.name}: {move_err}")
                        all_moved = False
                if all_moved:
                    try:
                        sub.rmdir()
                    except OSError as rmdir_err:
                        print(f"  WARNING: could not remove dir {sub.name}: {rmdir_err}")

        files = sorted(SCADA_RAW.glob("*.csv"))
        if files:
            print(f"CSV files ({len(files)}): {[f.name for f in files]}")
        else:
            print("Download command succeeded, but no CSV files were found.")
            print("Check dataset contents manually in data/raw/wind_scada/.")
    else:
        print("Kaggle download failed for all candidate dataset slugs.")
        if last_error:
            print("Last Kaggle error message:")
            print(last_error[:1200])
        print("Manual: https://www.kaggle.com/datasets/berkerisen/wind-turbine-scada-dataset")

In [ ]:
# 2-E  Download MIMII fan machine from Zenodo (large download)
# Only the fan machine is downloaded by default to save time/storage.
# Set MIMII_MACHINES to ["fan","pump","slider","valve"] for the full dataset.

import urllib.request
import urllib.error
import zipfile
from pathlib import Path

MIMII_RAW      = Path("data/raw/mimii")
MIMII_RAW.mkdir(parents=True, exist_ok=True)

MIMII_MACHINES = ["fan"]
MIMII_RECORD_ID = "3384388"
MIMII_PREFERRED_VARIANT = "6_dB"  # one of: "-6_dB", "0_dB", "6_dB"

ZENODO_BASES = [
    f"https://zenodo.org/records/{MIMII_RECORD_ID}/files",
    f"https://zenodo.org/record/{MIMII_RECORD_ID}/files",  # backward compatibility
]

DOWNLOAD_TIMEOUT = 30        # seconds per chunk read
CHUNK_SIZE       = 1 << 20   # 1 MB

for machine in MIMII_MACHINES:
    machine_dir = MIMII_RAW / machine
    if machine_dir.exists() and any(machine_dir.rglob("*.wav")):
        print(f"MIMII/{machine} already downloaded.")
        continue

    # Current Zenodo files are named like: 6_dB_fan.zip / 0_dB_fan.zip / -6_dB_fan.zip
    candidate_names = [
        f"{MIMII_PREFERRED_VARIANT}_{machine}.zip",
        f"0_dB_{machine}.zip",
        f"6_dB_{machine}.zip",
        f"-6_dB_{machine}.zip",
        f"{machine}.zip",  # legacy fallback
    ]

    # De-duplicate while preserving order
    seen = set()
    candidate_names = [n for n in candidate_names if not (n in seen or seen.add(n))]

    downloaded = False
    last_error = ""

    for base in ZENODO_BASES:
        if downloaded:
            break

        for fname in candidate_names:
            url = f"{base}/{fname}"
            zip_path = MIMII_RAW / fname
            print(f"Trying: {url}")

            try:
                resp = urllib.request.urlopen(url, timeout=DOWNLOAD_TIMEOUT)
                total = int(resp.headers.get("Content-Length", 0))
                got = 0

                with open(zip_path, "wb") as f:
                    while True:
                        chunk = resp.read(CHUNK_SIZE)
                        if not chunk:
                            break
                        f.write(chunk)
                        got += len(chunk)
                        if total > 0:
                            pct = min(int(got * 100 / total), 100)
                            print(
                                f"\r  {pct}% ({got / 1e6:.1f} / {total / 1e6:.1f} MB)",
                                end="",
                                flush=True,
                            )

                print("\nExtracting ...")
                with zipfile.ZipFile(zip_path, "r") as z:
                    z.extractall(MIMII_RAW)
                zip_path.unlink(missing_ok=True)

                downloaded = True
                print(f"MIMII/{machine} ready from {fname}.")
                break

            except (urllib.error.HTTPError, urllib.error.URLError, TimeoutError, OSError) as e:
                last_error = str(e)
                if zip_path.exists():
                    zip_path.unlink(missing_ok=True)
                print(f"  Failed: {e}")

    if not downloaded:
        print(f"\nDownload FAILED for MIMII/{machine}.")
        if last_error:
            print(f"Last error: {last_error}")
        print("Manual fallback:")
        print(f"  1) Open https://zenodo.org/records/{MIMII_RECORD_ID}")
        print(f"  2) Download one of: -6_dB_{machine}.zip / 0_dB_{machine}.zip / 6_dB_{machine}.zip")
        print(f"  3) Extract into {MIMII_RAW}/")

wav_count = len(list(MIMII_RAW.rglob("*.wav")))
print(f"\nTotal WAV files: {wav_count}")

In [ ]:
# 2-F  Verify downloads
from pathlib import Path

checks = {
    "C-MAPSS FD001 train": Path("data/raw/cmapss/train_FD001.txt"),
    "C-MAPSS FD001 test":  Path("data/raw/cmapss/test_FD001.txt"),
    "Wind SCADA CSV":      next(Path("data/raw/wind_scada").glob("*.csv"), Path("MISSING")),
    "MIMII WAV":           next(Path("data/raw/mimii").rglob("*.wav"), Path("MISSING")),
}

all_ok = True
for label, path in checks.items():
    exists = path.exists()
    print(f"  {'✓' if exists else '✗'} {label}: {path}")
    if not exists:
        all_ok = False

if all_ok:
    print("\nAll required data found — ready for preprocessing.")
else:
    print("\nSome data is missing. Please complete the download steps above before continuing.")


## 🔧 Section 3 — Data Preprocessing

- **C-MAPSS:** Per-engine chronological split, Kalman imputation, 13 informative sensors  
  Context window = 64 cycles, RUL horizon = 30 cycles  
- **Wind SCADA:** Multivariate CSV → slide windows (512 / 96)  
- **MIMII:** WAV → 40-MFCC frame matrix → slide windows (512 / 96)  

All processed files are saved as `.pt` PyTorch tensors under `data/processed/`.


In [ ]:
# 3-A  Import preprocessing module
from src.data.preprocessing import SCADAPreprocessor
from pathlib import Path

CMAPSS_RAW  = Path("data/raw/cmapss")
SCADA_RAW   = Path("data/raw/wind_scada")
MIMII_RAW   = Path("data/raw/mimii")
PROC_DIR    = Path("data/processed")


In [ ]:
# 3-B  C-MAPSS — all four subsets
import torch

preprocessor_cmapss = SCADAPreprocessor(
    lookback=CMAPSS_LOOKBACK,
    horizon=CMAPSS_HORIZON,
    train_ratio=0.70,
    val_ratio=0.15,
    normalization="standard",
    seed=SEED,
)

cmapss_data = {}
for subset in CMAPSS_SUBSETS:
    out_dir = PROC_DIR / "cmapss" / subset
    sentinel = out_dir / "processed_data.pt"

    if sentinel.exists():
        print(f"  Loading cached C-MAPSS {subset} …")
        cmapss_data[subset] = preprocessor_cmapss.load_processed(out_dir)
    else:
        print(f"\nPreprocessing C-MAPSS {subset} …")
        data = preprocessor_cmapss.process_cmapss(CMAPSS_RAW, subset=subset)
        preprocessor_cmapss.save_processed(data, out_dir)
        cmapss_data[subset] = data

    d = cmapss_data[subset]
    print(f"  {subset}: train {tuple(d['train_X'].shape)}  "
          f"val {tuple(d['val_X'].shape)}  test {tuple(d['test_X'].shape)}")


In [ ]:
# 3-C  Wind SCADA preprocessing
import pandas as pd
from pathlib import Path

preprocessor_scada = SCADAPreprocessor(
    lookback=LOOKBACK, horizon=HORIZON, train_ratio=0.70,
    val_ratio=0.15, normalization="standard", seed=SEED,
)

SCADA_OUT = PROC_DIR / "wind_scada"
sentinel  = SCADA_OUT / "processed_data.pt"

if sentinel.exists():
    print("Loading cached Wind SCADA …")
    scada_data = preprocessor_scada.load_processed(SCADA_OUT)
else:
    csv_files = sorted(Path("data/raw/wind_scada").glob("*.csv"))
    if not csv_files:
        raise FileNotFoundError("No CSV found in data/raw/wind_scada/ — run download step first.")
    csv_path = csv_files[0]
    print(f"Preprocessing Wind SCADA from {csv_path.name} …")

    # Detect timestamp column and drop non-numeric columns
    df_raw = pd.read_csv(csv_path, nrows=5)
    ts_col = next((c for c in df_raw.columns
                   if any(k in c.lower() for k in ["time", "date", "timestamp"])), None)

    scada_data = preprocessor_scada.process_generic_csv(
        csv_path,
        timestamp_col=ts_col,
        task="forecasting",
    )
    preprocessor_scada.save_processed(scada_data, SCADA_OUT)

d = scada_data
print(f"Wind SCADA: train {tuple(d['train_X'].shape)}  "
      f"val {tuple(d['val_X'].shape)}  test {tuple(d['test_X'].shape)}")


In [ ]:
# 3-D  MIMII audio → MFCC features → preprocessing
import numpy as np
import librosa
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

preprocessor_mimii = SCADAPreprocessor(
    lookback=LOOKBACK, horizon=HORIZON, train_ratio=0.70,
    val_ratio=0.15, normalization="standard", seed=SEED,
)

MIMII_OUT = PROC_DIR / "mimii"
sentinel  = MIMII_OUT / "processed_data.pt"

if sentinel.exists():
    print("Loading cached MIMII …")
    mimii_data = preprocessor_mimii.load_processed(MIMII_OUT)
else:
    print("Extracting MFCC features from MIMII audio files …")

    N_MFCC    = 40
    SR        = 16_000
    N_FFT     = 1024
    HOP       = 512
    MAX_FILES = 500   # limit for speed; increase for full run

    def extract_mfcc_timeline(wav_path: Path) -> np.ndarray:
        '''Return (n_frames, n_mfcc) feature matrix.'''
        y, sr = librosa.load(wav_path, sr=SR, mono=True)
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC,
                                     n_fft=N_FFT, hop_length=HOP)
        return mfcc.T  # (frames, 40)

    frames_list, labels_list = [], []

    all_wavs = sorted(MIMII_RAW.rglob("*.wav"))[:MAX_FILES]
    print(f"  Processing {len(all_wavs)} WAV files …")
    for wav in all_wavs:
        feat = extract_mfcc_timeline(wav)
        frames_list.append(feat)
        labels_list.append(0 if "normal" in str(wav) else 1)

    # Concatenate all frames into one long timeline (treat as single channel set)
    timeline = np.concatenate(frames_list, axis=0)  # (total_frames, 40)
    print(f"  MFCC timeline: {timeline.shape}")

    import pandas as pd
    import tempfile

    # Use NamedTemporaryFile to safely create the temp CSV (avoids mktemp race condition)
    with tempfile.NamedTemporaryFile(suffix=".csv", delete=False, mode="w") as _tf:
        tmp = Path(_tf.name)
        pd.DataFrame(timeline).to_csv(_tf, index=False)
    # File is closed/flushed here before passing to process_generic_csv

    try:
        mimii_data = preprocessor_mimii.process_generic_csv(tmp, task="forecasting")
    finally:
        tmp.unlink(missing_ok=True)

    preprocessor_mimii.save_processed(mimii_data, MIMII_OUT)

d = mimii_data
print(f"MIMII: train {tuple(d['train_X'].shape)}  "
      f"val {tuple(d['val_X'].shape)}  test {tuple(d['test_X'].shape)}")

In [ ]:
# 3-E  Checkpoint preprocessed data to Drive
import shutil
from pathlib import Path

if DRIVE_AVAILABLE and DRIVE_RESULTS_DIR:
    drive_proc = Path(DRIVE_RESULTS_DIR).parent / "processed"
    drive_proc.mkdir(parents=True, exist_ok=True)
    for dataset in ["cmapss", "wind_scada", "mimii"]:
        src = Path(f"data/processed/{dataset}")
        dst = drive_proc / dataset
        if src.exists():
            shutil.copytree(str(src), str(dst), dirs_exist_ok=True)
            print(f"  Checkpointed data/processed/{dataset} → Drive")
else:
    print("Drive not available — data stays local.")


## 🔬 Section 4 — Zero-Shot Experiments

All four models are evaluated **without any task-specific training**.  
PatchTST is trained on the training split first (it is a supervised baseline).

Each run saves a JSON result file under `results/zero_shot/`.


In [ ]:
# 4-A  Import model wrappers and metric utilities
from src.models import get_model
from src.evaluation.metrics import compute_all_metrics
import torch, numpy as np, json
from pathlib import Path
from datetime import datetime

RES_ZS = Path("results/zero_shot")
RES_ZS.mkdir(parents=True, exist_ok=True)

def load_dataset(name: str, subset: str = None):
    '''Return (X_train, y_train, X_test, y_test) as numpy arrays.'''
    if name == "cmapss":
        key = subset or "FD001"
        d = cmapss_data[key]
    elif name == "wind_scada":
        d = scada_data
    elif name == "mimii":
        d = mimii_data
    else:
        raise ValueError(f"Unknown dataset: {name}")
    to_np = lambda t: t.numpy() if isinstance(t, torch.Tensor) else t
    return (to_np(d["train_X"]), to_np(d["train_y"]),
            to_np(d["test_X"]),  to_np(d["test_y"]))


In [ ]:
# 4-B  Run zero-shot experiments
import traceback

def run_zero_shot(model_name: str, dataset_name: str,
                  X_train, y_train, X_test, y_test,
                  horizon: int) -> dict:
    model = get_model(model_name, device=DEVICE)

    if model_name == "patchtst":
        print(f"  PatchTST: fitting on training split first …")
        model.fit(torch.FloatTensor(X_train), torch.FloatTensor(y_train))
    else:
        model.load_model()

    results = model.zero_shot(torch.FloatTensor(X_test), horizon=horizon)
    preds = results["predictions"]          # (n_samples, horizon, channels)

    # Align shapes
    y = y_test
    if y.ndim == 1:
        # RUL scalar targets — compare with mean predictions
        preds_flat = preds.reshape(preds.shape[0], -1).mean(axis=1)
        y_flat     = y
    else:
        min_h = min(preds.shape[1], y.shape[1] if y.ndim > 1 else horizon)
        preds_flat = preds[:, :min_h].flatten()
        y_flat     = y[:, :min_h].flatten() if y.ndim > 1 else y.flatten()

    metrics = {
        "mae":  float(np.mean(np.abs(y_flat - preds_flat))),
        "rmse": float(np.sqrt(np.mean((y_flat - preds_flat) ** 2))),
        "mape": float(np.mean(np.abs((y_flat - preds_flat) /
                                     (np.abs(y_flat) + 1e-8))) * 100),
    }

    del model
    torch.cuda.empty_cache()
    return metrics


if not RUN_ZERO_SHOT:
    print("RUN_ZERO_SHOT=False — skipping section 4.")
else:
    zero_shot_results = {}

    # Dataset configs: (name, subset_or_None, horizon)
    dataset_cfgs = [
        ("cmapss",     "FD001", CMAPSS_HORIZON),
        ("wind_scada", None,    HORIZON),
        ("mimii",      None,    HORIZON),
    ]

    for ds_name, subset, h in dataset_cfgs:
        X_tr, y_tr, X_te, y_te = load_dataset(ds_name, subset)
        ds_label = f"{ds_name}/{subset}" if subset else ds_name

        for model_name in MODELS_ZERO_SHOT:
            key = f"{model_name}_{ds_label.replace('/', '_')}"
            json_path = RES_ZS / f"{key}.json"

            if json_path.exists():
                print(f"  [cached] {key}")
                with open(json_path) as f:
                    zero_shot_results[key] = json.load(f)
                continue

            print(f"\n{'='*60}")
            print(f"  Zero-shot: {model_name} on {ds_label}")
            print(f"{'='*60}")
            try:
                metrics = run_zero_shot(model_name, ds_name,
                                        X_tr, y_tr, X_te, y_te, h)
                record = {
                    "model": model_name, "dataset": ds_label,
                    "scenario": "zero_shot", "metrics": metrics,
                    "timestamp": datetime.now().isoformat(),
                }
                zero_shot_results[key] = record
                with open(json_path, "w") as f:
                    json.dump(record, f, indent=2)
                print(f"  → {metrics}")
            except Exception as e:
                print(f"  ERROR: {e}")
                traceback.print_exc()
                zero_shot_results[key] = {"model": model_name,
                                          "dataset": ds_label, "error": str(e)}

    print(f"\nZero-shot done. {len(zero_shot_results)} run(s) saved to {RES_ZS}")


## 🎯 Section 5 — Few-Shot LoRA Experiments

Only **MOMENT** and **Lag-Llama** support LoRA fine-tuning.  
Each model is fine-tuned using **1 %** of the training data (≥ 32 samples).  
LoRA config: `r=16`, `α=32`, `10 epochs`.


In [ ]:
# 5-A  Run few-shot LoRA experiments

RES_FS = Path("results/few_shot")
RES_FS.mkdir(parents=True, exist_ok=True)

def run_few_shot(model_name: str, dataset_name: str,
                 X_train, y_train, X_test, y_test,
                 horizon: int, train_ratio: float = TRAIN_RATIO) -> dict:
    n_few = max(int(len(X_train) * train_ratio), 32)
    X_few = torch.FloatTensor(X_train[:n_few])
    y_few = torch.FloatTensor(y_train[:n_few])

    model = get_model(model_name, device=DEVICE)
    model.load_model()
    model.few_shot_adapt(X_few, y_few,
                         epochs=LORA_EPOCHS, lr=LORA_LR,
                         lora_r=LORA_R, lora_alpha=LORA_ALPHA)

    results = model.predict(torch.FloatTensor(X_test), horizon=horizon)
    preds   = results["predictions"]

    y = y_test
    if y.ndim == 1:
        preds_flat = preds.reshape(preds.shape[0], -1).mean(axis=1)
        y_flat     = y
    else:
        min_h      = min(preds.shape[1], y.shape[1] if y.ndim > 1 else horizon)
        preds_flat = preds[:, :min_h].flatten()
        y_flat     = y[:, :min_h].flatten()

    metrics = {
        "mae":         float(np.mean(np.abs(y_flat - preds_flat))),
        "rmse":        float(np.sqrt(np.mean((y_flat - preds_flat) ** 2))),
        "mape":        float(np.mean(np.abs((y_flat - preds_flat) /
                                             (np.abs(y_flat) + 1e-8))) * 100),
        "train_samples": n_few,
    }

    del model
    torch.cuda.empty_cache()
    return metrics


if not RUN_FEW_SHOT:
    print("RUN_FEW_SHOT=False — skipping section 5.")
else:
    few_shot_results = {}

    dataset_cfgs_fs = [
        ("cmapss",     "FD001", CMAPSS_HORIZON),
        ("wind_scada", None,    HORIZON),
        ("mimii",      None,    HORIZON),
    ]

    for ds_name, subset, h in dataset_cfgs_fs:
        X_tr, y_tr, X_te, y_te = load_dataset(ds_name, subset)
        ds_label = f"{ds_name}/{subset}" if subset else ds_name

        for model_name in MODELS_FEW_SHOT:
            key       = f"{model_name}_{ds_label.replace('/', '_')}_few_shot"
            json_path = RES_FS / f"{key}.json"

            if json_path.exists():
                print(f"  [cached] {key}")
                with open(json_path) as f:
                    few_shot_results[key] = json.load(f)
                continue

            print(f"\n{'='*60}")
            print(f"  Few-shot LoRA: {model_name} on {ds_label}")
            print(f"{'='*60}")
            try:
                metrics = run_few_shot(model_name, ds_name,
                                       X_tr, y_tr, X_te, y_te, h)
                record = {
                    "model": model_name, "dataset": ds_label,
                    "scenario": "few_shot", "metrics": metrics,
                    "timestamp": datetime.now().isoformat(),
                }
                few_shot_results[key] = record
                with open(json_path, "w") as f:
                    json.dump(record, f, indent=2)
                print(f"  → {metrics}")
            except Exception as e:
                print(f"  ERROR: {e}")
                traceback.print_exc()
                few_shot_results[key] = {"model": model_name,
                                         "dataset": ds_label, "error": str(e)}

    print(f"\nFew-shot done. {len(few_shot_results)} run(s) saved to {RES_FS}")


## 🔄 Section 6 — Cross-Condition Transfer (C-MAPSS)

Train condition: **FD001** (one fault mode, moderate fan degradation)  
Transfer targets: **FD002**, **FD003**, **FD004** (more fan + HPC faults, higher variability)

For zero-shot TSFMs, no retraining is needed — the model simply runs inference on
the target condition directly (domain-agnostic feature extraction).


In [ ]:
# 6-A  Cross-condition experiments  (FD001 → FD002/003/004)
#      ALSO runs FD001 in-domain evaluation — needed for Table 5 FD001 baseline column

RES_CC = Path("results/cross_condition")
RES_CC.mkdir(parents=True, exist_ok=True)

SOURCE_SUBSET  = "FD001"
TARGET_SUBSETS = [s for s in CMAPSS_SUBSETS if s != SOURCE_SUBSET]

if not RUN_CROSS_CONDITION:
    print("RUN_CROSS_CONDITION=False — skipping section 6.")
else:
    cross_cond_results = {}

    X_src_tr_full, y_src_tr_full, X_src_te, y_src_te = load_dataset("cmapss", SOURCE_SUBSET)

    # ── Step A: FD001 in-domain baseline (Table 5 FD001 column) ──────────────
    print("\n── FD001 in-domain baseline (Table 5 FD001 column) ──")
    for model_name in MODELS_ZERO_SHOT:
        key       = f"{model_name}_FD001_to_FD001"
        json_path = RES_CC / f"{key}.json"

        if json_path.exists():
            print(f"  [cached] {key}")
            with open(json_path) as f:
                cross_cond_results[key] = json.load(f)
            continue

        print(f"\n  In-domain: {model_name} on FD001")
        try:
            model = get_model(model_name, device=DEVICE)
            if model_name == "patchtst":
                model.fit(torch.FloatTensor(X_src_tr_full),
                          torch.FloatTensor(y_src_tr_full))
            else:
                model.load_model()

            results = model.predict(torch.FloatTensor(X_src_te), horizon=CMAPSS_HORIZON)
            preds   = results["predictions"]

            y = y_src_te
            if y.ndim == 1:
                preds_flat = preds.reshape(preds.shape[0], -1).mean(axis=1)
                y_flat     = y
            else:
                min_h      = min(preds.shape[1], y.shape[1])
                preds_flat = preds[:, :min_h].flatten()
                y_flat     = y[:, :min_h].flatten()

            metrics = {
                "mae":  float(np.mean(np.abs(y_flat - preds_flat))),
                "rmse": float(np.sqrt(np.mean((y_flat - preds_flat) ** 2))),
            }
            record = {
                "model": model_name,
                "source": SOURCE_SUBSET, "target": SOURCE_SUBSET,
                "scenario": "cross_condition", "metrics": metrics,
                "timestamp": datetime.now().isoformat(),
            }
            cross_cond_results[key] = record
            with open(json_path, "w") as f:
                json.dump(record, f, indent=2)
            print(f"    → {metrics}")

            del model
            if torch.cuda.is_available(): torch.cuda.empty_cache()
        except Exception as e:
            print(f"    ERROR: {e}")
            traceback.print_exc()

    # ── Step B: FD001 → FD002 / FD003 / FD004 ────────────────────────────────
    for target in TARGET_SUBSETS:
        _, _, X_tgt_te, y_tgt_te = load_dataset("cmapss", target)

        for model_name in MODELS_ZERO_SHOT:
            key       = f"{model_name}_FD001_to_{target}"
            json_path = RES_CC / f"{key}.json"

            if json_path.exists():
                print(f"  [cached] {key}")
                with open(json_path) as f:
                    cross_cond_results[key] = json.load(f)
                continue

            print(f"\n  Cross-condition: {model_name}  FD001 → {target}")
            try:
                model = get_model(model_name, device=DEVICE)
                if model_name == "patchtst":
                    model.fit(torch.FloatTensor(X_src_tr_full),
                              torch.FloatTensor(y_src_tr_full))
                else:
                    model.load_model()

                results = model.predict(torch.FloatTensor(X_tgt_te), horizon=CMAPSS_HORIZON)
                preds   = results["predictions"]

                y = y_tgt_te
                if y.ndim == 1:
                    preds_flat = preds.reshape(preds.shape[0], -1).mean(axis=1)
                    y_flat     = y
                else:
                    min_h      = min(preds.shape[1], y.shape[1])
                    preds_flat = preds[:, :min_h].flatten()
                    y_flat     = y[:, :min_h].flatten()

                metrics = {
                    "mae":  float(np.mean(np.abs(y_flat - preds_flat))),
                    "rmse": float(np.sqrt(np.mean((y_flat - preds_flat) ** 2))),
                }
                record = {
                    "model": model_name,
                    "source": SOURCE_SUBSET, "target": target,
                    "scenario": "cross_condition", "metrics": metrics,
                    "timestamp": datetime.now().isoformat(),
                }
                cross_cond_results[key] = record
                with open(json_path, "w") as f:
                    json.dump(record, f, indent=2)
                print(f"    → {metrics}")

                del model
                if torch.cuda.is_available(): torch.cuda.empty_cache()
            except Exception as e:
                print(f"    ERROR: {e}")
                traceback.print_exc()
                cross_cond_results[key] = {"model": model_name,
                                           "source": SOURCE_SUBSET,
                                           "target": target, "error": str(e)}

    print(f"\nCross-condition done. {len(cross_cond_results)} run(s) saved to {RES_CC}")


## 📊 Section 7 — Results Aggregation & Tables

In [ ]:
# 7-A  Load all results into DataFrames
import pandas as pd, json
from pathlib import Path

def load_results_dir(res_dir: Path) -> pd.DataFrame:
    rows = []
    for f in sorted(res_dir.glob("*.json")):
        with open(f) as fp:
            r = json.load(fp)
        if "error" in r:
            continue
        m = r.get("metrics", {})
        row = {
            "scenario":  r.get("scenario", res_dir.name),
            "model":     r.get("model"),
            "dataset":   r.get("dataset", r.get("target")),
            "mae":       m.get("mae"),
            "rmse":      m.get("rmse"),
            "mape":      m.get("mape"),
        }
        if "source" in r:
            row["source"]  = r["source"]
            row["target"]  = r["target"]
        rows.append(row)
    return pd.DataFrame(rows)

df_zs = load_results_dir(Path("results/zero_shot"))
df_fs = load_results_dir(Path("results/few_shot"))
df_cc = load_results_dir(Path("results/cross_condition"))
df_all = pd.concat([df_zs, df_fs, df_cc], ignore_index=True)

print(f"Total result records: {len(df_all)}")
print(df_all.head(20).to_string(index=False))


In [ ]:
# 7-B  Zero-shot pivot table (MAE)
if not df_zs.empty:
    pivot_zs = df_zs.pivot_table(
        index="model", columns="dataset", values="mae", aggfunc="mean"
    )
    print("\n=== Zero-Shot MAE ===")
    print(pivot_zs.round(4).to_string())
else:
    print("No zero-shot results yet.")


In [ ]:
# 7-C  Few-shot pivot table (MAE)
if not df_fs.empty:
    pivot_fs = df_fs.pivot_table(
        index="model", columns="dataset", values="mae", aggfunc="mean"
    )
    print("\n=== Few-Shot (LoRA 1%) MAE ===")
    print(pivot_fs.round(4).to_string())
else:
    print("No few-shot results yet.")


In [ ]:
# 7-D  Cross-condition table (MAE, FD001 → FD00x)
if not df_cc.empty:
    pivot_cc = df_cc.pivot_table(
        index="model", columns="target", values="mae", aggfunc="mean"
    )
    print("\n=== Cross-Condition MAE (source: FD001) ===")
    print(pivot_cc.round(4).to_string())
else:
    print("No cross-condition results yet.")


## ⏱️ Section 8 — Inference Benchmarking & Results Export

Profiles **inference latency**, **peak GPU memory**, and **parameter count** for each model.  
Then exports all aggregated results as clean **CSV files** under `results/tables/` for later use in article writing.

In [ ]:
# 8-A  Inference latency & GPU memory profiling
import time, json
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from src.models import get_model

PROFILE_DIR = Path("results/tables")
PROFILE_DIR.mkdir(parents=True, exist_ok=True)

# Fixed profiling batch: 32 C-MAPSS test samples
_, _, X_prof_np, _ = load_dataset("cmapss", "FD001")
X_prof = torch.FloatTensor(X_prof_np[:32])

WARMUP_RUNS = 2
TIMING_RUNS = 5

profile_results = {}

for model_name in MODELS_ZERO_SHOT:
    print(f"\nProfiling {model_name} ...")
    cached_path = PROFILE_DIR / f"profile_{model_name}.json"
    if cached_path.exists():
        with open(cached_path) as f:
            profile_results[model_name] = json.load(f)
        print(f"  [cached]")
        continue
    try:
        model = get_model(model_name, device=DEVICE)
        if model_name == "patchtst":
            X_tr_p, y_tr_p, *_ = load_dataset("cmapss", "FD001")
            model.fit(torch.FloatTensor(X_tr_p[:64]), torch.FloatTensor(y_tr_p[:64]))
        else:
            model.load_model()

        _inner = getattr(model, "model", None)
        n_params = sum(p.numel() for p in _inner.parameters()) if _inner is not None else 0

        # Warmup
        for _ in range(WARMUP_RUNS):
            _ = model.predict(X_prof, horizon=CMAPSS_HORIZON)

        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
            torch.cuda.synchronize()

        # Timed runs
        t0 = time.perf_counter()
        for _ in range(TIMING_RUNS):
            _ = model.predict(X_prof, horizon=CMAPSS_HORIZON)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        elapsed_ms = (time.perf_counter() - t0) / TIMING_RUNS * 1000

        peak_mem_mb = (torch.cuda.max_memory_allocated() / 1e6
                       if torch.cuda.is_available() else float("nan"))

        rec = {"model": model_name,
               "params_M":    round(n_params / 1e6, 1),
               "latency_ms":  round(elapsed_ms, 1),
               "peak_gpu_mb": round(peak_mem_mb, 0),
               "batch_size":  int(X_prof.shape[0])}
        profile_results[model_name] = rec
        with open(cached_path, "w") as f:
            json.dump(rec, f, indent=2)
        print(f"  params: {rec['params_M']} M  |  latency: {rec['latency_ms']} ms  "
              f"|  peak GPU: {rec['peak_gpu_mb']} MB")

        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    except Exception as e:
        print(f"  ERROR: {e}")
        profile_results[model_name] = {"model": model_name, "error": str(e)}

# Save efficiency results as CSV
eff_rows = [v for v in profile_results.values() if "error" not in v]
df_eff = pd.DataFrame(eff_rows)
df_eff.to_csv(PROFILE_DIR / "efficiency_summary.csv", index=False)

print("\n=== Deployment Efficiency ===")
print(f"{'Model':<12} {'Params (M)':>11} {'Latency (ms)':>13} {'Peak GPU (MB)':>14}")
print("-" * 55)
for m in MODELS_ZERO_SHOT:
    r2 = profile_results.get(m, {})
    if not r2 or "error" in r2:
        print(f"  {m:<12}  (error)")
    else:
        print(f"  {m:<12} {r2['params_M']:>10.1f} "
              f" {r2['latency_ms']:>12.1f}  {r2['peak_gpu_mb']:>13.0f}")
print(f"\nSaved → results/tables/efficiency_summary.csv")

In [ ]:
# 8-B  Export all aggregated results as CSV files
#      These CSVs contain everything needed to build article tables later.
import pandas as pd
from pathlib import Path

TABLES_DIR = Path("results/tables")
TABLES_DIR.mkdir(parents=True, exist_ok=True)

# ── 1. Zero-shot summary (model × dataset × metrics) ──────────────────────
if not df_zs.empty:
    # Pivot: rows=model, columns=dataset, values=mae
    pivot_zs_mae = df_zs.pivot_table(index="model", columns="dataset",
                                      values="mae", aggfunc="mean")
    pivot_zs_mae["mean"] = pivot_zs_mae.mean(axis=1)
    pivot_zs_mae.to_csv(TABLES_DIR / "zero_shot_mae.csv")

    # Full detail (all metrics)
    df_zs.to_csv(TABLES_DIR / "zero_shot_full.csv", index=False)
    print(f"Zero-shot: {len(df_zs)} runs → zero_shot_mae.csv + zero_shot_full.csv")
    print(pivot_zs_mae.round(4).to_string())
else:
    print("No zero-shot results to export.")

# ── 2. Few-shot summary ───────────────────────────────────────────────────
if not df_fs.empty:
    pivot_fs_mae = df_fs.pivot_table(index="model", columns="dataset",
                                      values="mae", aggfunc="mean")
    pivot_fs_mae["mean"] = pivot_fs_mae.mean(axis=1)
    pivot_fs_mae.to_csv(TABLES_DIR / "few_shot_mae.csv")
    df_fs.to_csv(TABLES_DIR / "few_shot_full.csv", index=False)
    print(f"\nFew-shot: {len(df_fs)} runs → few_shot_mae.csv + few_shot_full.csv")
    print(pivot_fs_mae.round(4).to_string())
else:
    print("No few-shot results to export.")

# ── 3. Zero-shot vs Few-shot comparison (for LoRA-capable models) ─────────
if not df_zs.empty and not df_fs.empty:
    rows_cmp = []
    for m in MODELS_FEW_SHOT:
        for ds in df_zs["dataset"].unique():
            zs_val = df_zs[(df_zs["model"] == m) & (df_zs["dataset"] == ds)]["mae"].mean()
            fs_val = df_fs[(df_fs["model"] == m) & (df_fs["dataset"] == ds)]["mae"].mean()
            if pd.notna(zs_val) or pd.notna(fs_val):
                improvement = ((zs_val - fs_val) / zs_val * 100) if pd.notna(zs_val) and pd.notna(fs_val) and zs_val > 0 else None
                rows_cmp.append({"model": m, "dataset": ds,
                                 "zero_shot_mae": zs_val, "few_shot_mae": fs_val,
                                 "improvement_pct": improvement})
    df_cmp = pd.DataFrame(rows_cmp)
    df_cmp.to_csv(TABLES_DIR / "zs_vs_fs_comparison.csv", index=False)
    print(f"\nZS vs FS comparison → zs_vs_fs_comparison.csv")
    print(df_cmp.round(4).to_string(index=False))

# ── 4. Cross-condition summary ────────────────────────────────────────────
if not df_cc.empty:
    pivot_cc_mae = df_cc.pivot_table(index="model", columns="target",
                                      values="mae", aggfunc="mean")
    # Reorder columns: FD001, FD002, FD003, FD004
    col_order = [c for c in ["FD001", "FD002", "FD003", "FD004"] if c in pivot_cc_mae.columns]
    pivot_cc_mae = pivot_cc_mae[col_order]
    pivot_cc_mae.to_csv(TABLES_DIR / "cross_condition_mae.csv")
    df_cc.to_csv(TABLES_DIR / "cross_condition_full.csv", index=False)
    print(f"\nCross-condition: {len(df_cc)} runs → cross_condition_mae.csv + cross_condition_full.csv")
    print(pivot_cc_mae.round(4).to_string())
else:
    print("No cross-condition results to export.")

# ── 5. Combined master CSV ────────────────────────────────────────────────
df_all.to_csv(TABLES_DIR / "all_results.csv", index=False)

print(f"\n{'='*60}")
print(f"All CSV files saved to: results/tables/")
for f in sorted(TABLES_DIR.glob("*.csv")):
    print(f"  {f.name}")
print(f"{'='*60}")

## 📈 Section 9 — Generate Figures

In [ ]:
# 9-A  Bar chart — Zero-shot MAE by model × dataset
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({"font.size": 11, "figure.dpi": 150})

FIGS_DIR = Path("results/figures")
FIGS_DIR.mkdir(parents=True, exist_ok=True)

if not df_zs.empty:
    fig, ax = plt.subplots(figsize=(9, 4))
    datasets_plot = df_zs["dataset"].unique()
    models_plot   = [m for m in MODELS_ZERO_SHOT if m in df_zs["model"].values]

    x = range(len(models_plot))
    width = 0.8 / max(len(datasets_plot), 1)
    colors = plt.cm.tab10.colors

    for i, ds in enumerate(datasets_plot):
        vals = [
            df_zs[(df_zs["model"] == m) & (df_zs["dataset"] == ds)]["mae"].mean()
            for m in models_plot
        ]
        offset = (i - len(datasets_plot) / 2 + 0.5) * width
        bars = ax.bar([xi + offset for xi in x], vals, width,
                      label=ds, color=colors[i % len(colors)], alpha=0.85)

    ax.set_xticks(list(x))
    ax.set_xticklabels(models_plot, rotation=15, ha="right")
    ax.set_ylabel("MAE")
    ax.set_title("Zero-Shot MAE by Model and Dataset")
    ax.legend(title="Dataset", bbox_to_anchor=(1.01, 1), loc="upper left")
    fig.tight_layout()
    fig.savefig(FIGS_DIR / "zero_shot_mae_bar.pdf", bbox_inches="tight")
    fig.savefig(FIGS_DIR / "zero_shot_mae_bar.png", bbox_inches="tight")
    plt.show()
    print("Saved zero_shot_mae_bar.pdf / .png")
else:
    print("No zero-shot data to plot.")


In [ ]:
# 9-B  Zero-shot vs Few-shot MAE comparison (MOMENT + Lag-Llama)
import matplotlib.pyplot as plt
import numpy as np

if not df_zs.empty and not df_fs.empty:
    fs_models = [m for m in MODELS_FEW_SHOT if m in df_fs["model"].values]
    fig, axes = plt.subplots(1, len(fs_models), figsize=(5*len(fs_models), 4),
                             sharey=False)
    if len(fs_models) == 1:
        axes = [axes]

    for ax, model in zip(axes, fs_models):
        zs_row = df_zs[df_zs["model"] == model].set_index("dataset")["mae"]
        fs_row = df_fs[df_fs["model"] == model].set_index("dataset")["mae"]

        common_ds = sorted(set(zs_row.index) & set(fs_row.index))
        if not common_ds:
            ax.set_title(f"{model}\n(no common datasets)")
            continue

        x = np.arange(len(common_ds))
        ax.bar(x - 0.2, [zs_row.get(d, np.nan) for d in common_ds],
               0.35, label="Zero-shot", color="steelblue", alpha=0.85)
        ax.bar(x + 0.2, [fs_row.get(d, np.nan) for d in common_ds],
               0.35, label="Few-shot", color="coral", alpha=0.85)
        ax.set_xticks(x)
        ax.set_xticklabels(common_ds, rotation=15, ha="right")
        ax.set_ylabel("MAE")
        ax.set_title(f"{model.upper()}: ZS vs FS (1%)")
        ax.legend()

    fig.tight_layout()
    fig.savefig(FIGS_DIR / "zs_vs_fs_comparison.pdf", bbox_inches="tight")
    fig.savefig(FIGS_DIR / "zs_vs_fs_comparison.png", bbox_inches="tight")
    plt.show()
    print("Saved zs_vs_fs_comparison.pdf / .png")
else:
    print("Need both zero-shot and few-shot results to plot comparison.")


In [ ]:
# 9-C  Cross-condition heatmap (MAE)
import matplotlib.pyplot as plt
import seaborn as sns

if not df_cc.empty:
    models_cc = [m for m in MODELS_ZERO_SHOT if m in df_cc["model"].values]
    targets   = sorted(df_cc["target"].unique())

    mat = []
    for model in models_cc:
        row = []
        for tgt in targets:
            val = df_cc[(df_cc["model"] == model) & (df_cc["target"] == tgt)]["mae"].mean()
            row.append(val)
        mat.append(row)

    import numpy as np
    mat = np.array(mat, dtype=float)

    fig, ax = plt.subplots(figsize=(max(4, len(targets) * 1.5), max(3, len(models_cc))))
    sns.heatmap(mat, annot=True, fmt=".4f",
                xticklabels=targets, yticklabels=models_cc,
                cmap="YlOrRd", ax=ax,
                cbar_kws={"label": "MAE"})
    ax.set_title("Cross-Condition MAE: C-MAPSS FD001 → FD00x")
    ax.set_xlabel("Target Condition")
    ax.set_ylabel("Model")
    fig.tight_layout()
    fig.savefig(FIGS_DIR / "cross_condition_heatmap.pdf", bbox_inches="tight")
    fig.savefig(FIGS_DIR / "cross_condition_heatmap.png", bbox_inches="tight")
    plt.show()
    print("Saved cross_condition_heatmap.pdf / .png")
else:
    print("No cross-condition data to plot.")


## 💾 Section 10 — Export to Google Drive

In [ ]:
# 10-A  Copy all results + figures + tables to Drive
import shutil
from pathlib import Path
from datetime import datetime

if DRIVE_AVAILABLE and DRIVE_RESULTS_DIR:
    drive_root = Path(DRIVE_RESULTS_DIR)
    drive_root.mkdir(parents=True, exist_ok=True)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    for src_rel in ["results/zero_shot", "results/few_shot",
                    "results/cross_condition", "results/tables", "results/figures"]:
        src = Path(src_rel)
        if src.exists():
            dst = drive_root / src.relative_to("results")
            shutil.copytree(str(src), str(dst), dirs_exist_ok=True)
            print(f"  Copied {src_rel} → Drive")

    # Also save a timestamped summary CSV
    if not df_all.empty:
        summary_path = drive_root / f"summary_{timestamp}.csv"
        df_all.to_csv(summary_path, index=False)
        print(f"  Summary CSV → {summary_path}")
else:
    print("Drive not available — results remain in /content/tsfm-pdm-bench/results/")


In [ ]:
# 10-B  Final summary printout
print("=" * 70)
print("  EXPERIMENT SUMMARY")
print("=" * 70)

if not df_zs.empty:
    print(f"\nZero-shot runs:    {len(df_zs)}")
    print(f"  Avg MAE (all):    {df_zs['mae'].mean():.4f}")
if not df_fs.empty:
    print(f"\nFew-shot runs:     {len(df_fs)}")
    print(f"  Avg MAE (all):    {df_fs['mae'].mean():.4f}")
if not df_cc.empty:
    print(f"\nCross-cond runs:   {len(df_cc)}")
    print(f"  Avg MAE (FD001→): {df_cc['mae'].mean():.4f}")

print("\nOutput locations:")
print("  data/processed/          ← preprocessed .pt tensors")
print("  results/zero_shot/       ← per-run JSON files")
print("  results/few_shot/        ← per-run JSON files")
print("  results/cross_condition/ ← per-run JSON files")
print("  results/tables/          ← aggregated CSV summaries + efficiency profiles")
print("  results/figures/         ← PDF + PNG figures")
if DRIVE_AVAILABLE and DRIVE_RESULTS_DIR:
    print(f"  Google Drive:              {DRIVE_RESULTS_DIR}")
print("\nAll done — you have everything needed to write the article!")